# 🤖 StrategyAI: RAG Chatbot (Gemini Version)
**Run each cell in order.**

| Component | Model |
|-----------|-------|
| Chat / Answer Generation | `gemini-2.5-flash` |
| Embeddings | `models/gemini-embedding-001` |
| Vector Store | ChromaDB (local, persistent) |

> ⚠️ **Requires:** Google Gemini API key — [get one free at aistudio.google.com](https://aistudio.google.com/app/apikey)

## 📦 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install -q \
    google-generativeai chromadb ipywidgets
print('✅ All packages installed')

✅ All packages installed


## 🔑 Set Your Gemini API Key

In [51]:
import os
import google.generativeai as genai

# ── EDIT THIS ──────────────────────────────────────────────
GEMINI_API_KEY = 'YOUR_API_KEY_HERE'   # <-- paste your key from aistudio.google.com
# ───────────────────────────────────────────────────────────

# On Google Colab you can use secrets instead:
# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)

# Quick connectivity test
test = genai.GenerativeModel('gemini-2.5-flash-lite').generate_content('Say OK')
print('✅ Gemini connected:', test.text.strip())

✅ Gemini connected: OK


In [42]:
# Removed: google-genai (new SDK) conflicted with google-generativeai (old SDK)
# This notebook uses google-generativeai throughout — no changes needed here.
print('✅ Using google-generativeai SDK throughout')

In [43]:
# Removed: new SDK model listing cell — caused genai.GenerativeModel AttributeError
# The embedding model in use is: models/gemini-embedding-001
print('✅ Embedding model: models/gemini-embedding-001')

models/gemini-embedding-001


## 📄 Document Processor (Paragraph Chunking)

In [5]:
import re
import hashlib
import json
from pathlib import Path
from dataclasses import dataclass, field

@dataclass
class Chunk:
    text: str
    metadata: dict = field(default_factory=dict)

    @property
    def chunk_id(self):
        return hashlib.md5(self.text.encode()).hexdigest()[:12]


def chunk_paragraph(text, min_words=30, max_words=300):
    raw = re.split(r'\n{2,}', text)
    chunks, buffer = [], ''
    for para in raw:
        para = para.strip()
        if not para or len(para.split()) < 10:
            continue
        bw = len(buffer.split()) if buffer else 0
        pw = len(para.split())
        if bw + pw <= max_words:
            buffer = (buffer + '\n\n' + para).strip()
        else:
            if buffer and bw >= min_words:
                chunks.append(buffer)
            buffer = para
    if buffer and len(buffer.split()) >= min_words:
        chunks.append(buffer)
    return chunks


def process_directory(directory):
    all_chunks = []
    files = sorted(Path(directory).rglob('*.txt'))
    for fp in files:
        text = fp.read_text(encoding='utf-8', errors='ignore')
        for i, chunk_text in enumerate(chunk_paragraph(text)):
            if len(chunk_text.split()) < 15:
                continue
            all_chunks.append(Chunk(
                text=chunk_text,
                metadata={
                    'source': fp.name,
                    'title': fp.stem.replace('_', ' ').title(),
                    'chunk_index': i,
                    'word_count': len(chunk_text.split())
                }
            ))
    print(f'📊 {len(all_chunks)} chunks from {len(files)} documents')
    return all_chunks

print('✅ Document processor ready')

✅ Document processor ready


## 📝 Generate 50+ Business Strategy Documents

In [6]:
import csv
from pathlib import Path

DOCS_DIR = Path('./rag_docs')
DOCS_DIR.mkdir(exist_ok=True)

# ── Load from strategy_documents.csv (place it next to this notebook) ────────
CSV_PATH = Path('./strategy_documents.csv')

if not CSV_PATH.exists():
    raise FileNotFoundError(
        "strategy_documents.csv not found. "
        "Download it from the project files and place it next to this notebook."
    )

doc_count = 0
with open(CSV_PATH, encoding='utf-8') as f:
    for row in csv.DictReader(f):
        out_path = DOCS_DIR / row['source']
        out_path.write_text(
            f"{row['title']}\n{'=' * len(row['title'])}\n\n{row['content']}",
            encoding='utf-8'
        )
        doc_count += 1

total = len(list(DOCS_DIR.glob('*.txt')))
print(f'✅ Loaded {doc_count} documents from CSV → {total} .txt files in {DOCS_DIR}/')


✅ Generated 50 documents in rag_docs/


## 🗄️ Build ChromaDB Vector Store with Gemini Embeddings

In [7]:
import chromadb
import time

# Gemini embedding function for ChromaDB
class GeminiEmbeddingFunction(chromadb.EmbeddingFunction):
    def __call__(self, input):
        embeddings = []
        for text in input:
            result = genai.embed_content(
                model='models/gemini-embedding-001',
                content=text,
                task_type='retrieval_document'
            )
            embeddings.append(result['embedding'])
        return embeddings

def embed_query(text):
    """Embed a single query string."""
    result = genai.embed_content(
        model='models/gemini-embedding-001',
        content=text,
        task_type='retrieval_query'
    )
    return result['embedding']

def embed_queries(texts):
    """Embed multiple query strings."""
    return [embed_query(t) for t in texts]

# ChromaDB setup
chroma = chromadb.PersistentClient(path='./chroma_db')
try:
    chroma.delete_collection('rag_chatbot')  # reset if re-running
except:
    pass
collection = chroma.get_or_create_collection(
    name='rag_chatbot',
    metadata={'hnsw:space': 'cosine'}
)

# Ingest with Gemini embeddings
chunks = process_directory('./rag_docs')
existing = set(collection.get()['ids'])
new_chunks = [c for c in chunks if c.chunk_id not in existing]

print(f'Embedding {len(new_chunks)} chunks with Gemini text-embedding-004...')
BATCH = 20   # Gemini free tier: be conservative
for i in range(0, len(new_chunks), BATCH):
    batch = new_chunks[i:i+BATCH]
    embs = []
    for c in batch:
        # r = genai.embed_content(
        #     model='models/gemini-embedding-001',
        #     content=c.text,
        #     task_type='retrieval_document'
        r = genai.embed_content(
            model='models/gemini-embedding-001',
            content=c.text,
            task_type='retrieval_document'
        )
        embs.append(r['embedding'])
    collection.upsert(
        ids=[c.chunk_id for c in batch],
        embeddings=embs,
        documents=[c.text for c in batch],
        metadatas=[c.metadata for c in batch]
    )
    print(f'  Batch {i//BATCH+1}/{(len(new_chunks)-1)//BATCH+1} done ({i+len(batch)}/{len(new_chunks)})')
    time.sleep(0.5)   # rate limit safety

print(f'\n✅ Vector store ready: {collection.count()} chunks indexed')

📊 50 chunks from 50 documents
Embedding 50 chunks with Gemini text-embedding-004...
  Batch 1/3 done (20/50)
  Batch 2/3 done (40/50)
  Batch 3/3 done (50/50)

✅ Vector store ready: 50 chunks indexed


## 🧠 Multi-Query RAG Engine + Conversation Memory

In [46]:
import google.generativeai as genai

gemini_chat_model = genai.GenerativeModel('gemini-2.5-flash')

# ── Conversation Memory ───────────────────────────────────────────────────────
conversation_history = []
MAX_TURNS = 8

def add_to_memory(user_msg, assistant_msg):
    conversation_history.append({'role': 'user', 'content': user_msg})
    conversation_history.append({'role': 'assistant', 'content': assistant_msg})
    while len(conversation_history) > MAX_TURNS * 2:
        conversation_history.pop(0)

def get_history_text():
    if not conversation_history:
        return '(No prior conversation)'
    lines = []
    for t in conversation_history[-8:]:
        role = 'User' if t['role'] == 'user' else 'Assistant'
        lines.append(f"{role}: {t['content'][:250]}")
    return '\n'.join(lines)


# ── Multi-Query Generation (Advanced Feature) ─────────────────────────────────
def generate_query_variations(query, n=3):
    prompt = f"""Generate {n} different search query variations for the question below.
Each variation should use different phrasing, synonyms, or related angles.
Output ONLY a JSON array of strings, nothing else.

Question: {query}

Output: ["variation 1", "variation 2", "variation 3"]"""
    try:
        resp = gemini_chat_model.generate_content(prompt)
        raw = resp.text.strip()
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            variations = json.loads(match.group())
            return [query] + [v for v in variations if v != query][:n]
    except:
        pass
    return [query]


# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def multi_query_retrieve(queries, top_k_each=8, final_k=5):
    all_candidates = {}
    rank_lists = []
    n = min(top_k_each, collection.count())
    for q in queries:
        emb = embed_query(q)
        # results = collection.query(
        #     query_embeddings=[emb], n_results=n,
        #     include=['documents', 'metadatas', 'distances', 'ids']
        # )
        results = collection.query(
            query_embeddings=[emb], n_results=n,
            include=['documents', 'metadatas', 'distances']  # remove 'ids'
        )
        ranked = []
        for doc, meta, dist, id_ in zip(
            results['documents'][0], results['metadatas'][0],
            results['distances'][0], results['ids'][0]
        ):
            all_candidates[id_] = {'text': doc, 'metadata': meta, 'score': round(1-dist, 4)}
            ranked.append(id_)
        rank_lists.append(ranked)
        time.sleep(0.2)   # rate limit safety
    # RRF
    K, rrf = 60, {}
    for ranked in rank_lists:
        for rank, id_ in enumerate(ranked):
            rrf[id_] = rrf.get(id_, 0) + 1 / (K + rank + 1)
    top_ids = sorted(rrf, key=rrf.get, reverse=True)[:final_k]
    return [all_candidates[i] for i in top_ids if i in all_candidates]


# ── Main Ask Function ─────────────────────────────────────────────────────────
def ask(query, verbose=True):
    # Resolve short follow-ups with prior context
    FOLLOWUPS = ['tell me more', 'can you elaborate', 'explain that', 'why', 'how', 'example', 'and?']
    q_lower = query.lower().strip()
    if conversation_history and (len(query.split()) <= 5 or any(q_lower.startswith(f) for f in FOLLOWUPS)):
        last_user = next((t['content'] for t in reversed(conversation_history) if t['role'] == 'user'), None)
        if last_user and last_user.lower().strip() != q_lower:
            query = f"{query} (context: {last_user[:120]})"

    # Multi-query + RRF
    queries = generate_query_variations(query)
    chunks  = multi_query_retrieve(queries)

    # Build context string
    context_parts = []
    for i, c in enumerate(chunks, 1):
        m = c['metadata']
        context_parts.append(f"[Chunk {i}] Source: {m.get('source','?')} | Relevance: {c['score']:.3f}\n{c['text']}")
    context = '\n\n---\n\n'.join(context_parts)

    prompt = f"""You are an expert Business Strategy & Management assistant.
Answer the question using ONLY the provided context. Be precise and structured.
At the end, list which sources you used.

CONTEXT:
{context}

CONVERSATION HISTORY:
{get_history_text()}

QUESTION: {query}

Answer (end with: Sources: [filenames used]):"""

    resp   = gemini_chat_model.generate_content(prompt)
    answer = resp.text.strip()
    sources = [
        {'source': c['metadata'].get('source'), 'score': c['score'],
         'title': c['metadata'].get('title', '')}
        for c in chunks
    ]

    add_to_memory(query, answer)

    if verbose:
        print(f'\n🔍 Queries: {queries}\n')
        print('─' * 70)
        print(answer)
        print('─' * 70)
        print('📚 Sources:', [s['source'] for s in sources])

    return answer, sources, queries

print('✅ Gemini RAG engine ready!')

✅ Gemini RAG engine ready!


## 💬 Quick Test

In [10]:
answer, sources, queries = ask("What is Porter's Five Forces framework?")


🔍 Queries: ["What is Porter's Five Forces framework?", "Porter's Five Forces explanation", "What is the purpose of Porter's 5 Forces model", "Define Porter's Five Forces framework"]

──────────────────────────────────────────────────────────────────────
Porter's Five Forces framework, introduced by Michael Porter in 1979, is an analytical tool used to assess industry competitiveness. It helps to identify industry attractiveness and defensible competitive positions before making major investment decisions.

The framework comprises five forces:
1.  **Threat of New Entrants**: This force examines the ease or difficulty for new competitors to enter an industry. High barriers to entry, such as significant capital requirements, economies of scale, strong brand loyalty, and regulation, protect existing companies.
2.  **Bargaining Power of Suppliers**: Suppliers gain leverage when they are concentrated or offer unique products. Companies can mitigate this power by diversifying their supplier 

## 💬 Multi-Turn Conversation Test

In [47]:
conversation_history.clear()

print('=== Turn 1 ===')
ask('What is Blue Ocean Strategy?')

print('\n=== Turn 2 — follow-up ===')
ask('Can you give me a real company example?')

=== Turn 1 ===

🔍 Queries: ['What is Blue Ocean Strategy?', 'define Blue Ocean Strategy', 'explain the concept of Blue Ocean Strategy', 'key principles of Blue Ocean Strategy']

──────────────────────────────────────────────────────────────────────
Blue Ocean Strategy, as proposed by Kim and Mauborgne (2005), advocates for companies to create uncontested market space rather than competing in existing, overcrowded markets (red oceans). Red oceans are characterized by known rules and fierce competition, while blue oceans are untapped spaces where demand is created. A core concept within this strategy is Value Innovation, which involves simultaneously pursuing differentiation and low cost to make competition irrelevant. The ERRC Framework (Eliminate, Reduce, Raise, Create) is a tool used to achieve this by identifying factors to eliminate or reduce, and factors to raise or create that the industry has never offered. An example is Cirque du Soleil, which eliminated animals and star perform

('The provided context does not contain information about real company examples of Blue Ocean Strategy.\n\nSources: [None]',
 [{'source': 'entrepreneurship.txt',
   'score': 0.6572,
   'title': 'Entrepreneurship'},
  {'source': 'competitive_advantage.txt',
   'score': 0.6469,
   'title': 'Competitive Advantage'},
  {'source': 'data_analytics_business.txt',
   'score': 0.646,
   'title': 'Data Analytics Business'},
  {'source': 'business_model_canvas.txt',
   'score': 0.6692,
   'title': 'Business Model Canvas'},
  {'source': 'mergers_acquisitions.txt',
   'score': 0.6395,
   'title': 'Mergers Acquisitions'}],
 ['Can you give me a real company example?',
  'Provide a real-world business case study.',
  'Show me a tangible company illustration.',
  'Give me a practical example of a business in operation.'])

## 🎛️ Interactive Chat Widget
Full chatbot UI inside Jupyter- type your question and click **Send**!

In [29]:
!pip install gradio

  You can safely remove it manually.



   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   ---- ----------------------------------- 2.9/24.2 MB 16.2 MB/s eta 0:00:02
   ---------- ----------------------------- 6.3/24.2 MB 16.1 MB/s eta 0:00:02
   ---------------- ----------------------- 10.0/24.2 MB 16.4 MB/s eta 0:00:01
   ---------------------- ----------------- 13.4/24.2 MB 16.4 MB/s eta 0:00:01
   --------------------------- ------------ 16.5/24.2 MB 16.3 MB/s eta 0:00:01
   -------------------------------- ------- 19.9/24.2 MB 16.3 MB/s eta 0:00:01
   -------------------------------------- - 23.3/24.2 MB 16.3 MB/s eta 0:00:01
   ---------------------------------------- 24.2/24.2 MB 15.4 MB/s eta 0:00:00

  Attempting uninstall: brotli

    Found existing installation: Brotli 1.0.9

    Uninstalling Brotli-1.0.9:

   --- ------------------------------------  1/13 [brotli]
   --- ------------------------------------  1/13 [brotli]
   --- ------------------------------------  1/13 [brotli]
   --- ---

In [33]:
!pip install flask flask-cors pyngrok


   ---------------------------------------- 2/2 [flask-cors]



In [15]:
# # Flask Backend  (run this, keep it running)
# import threading
# from flask import Flask, request, jsonify
# from flask_cors import CORS

# flask_app = Flask(__name__)
# CORS(flask_app)          # allow the HTML file to call this from any origin

# @flask_app.route("/chat", methods=["POST"])
# def chat_endpoint():
#     data = request.json
#     answer, sources, queries = ask(data["message"], verbose=False)
#     return jsonify({"answer": answer, "sources": sources, "queries": queries})

# @flask_app.route("/reset", methods=["POST"])
# def reset_endpoint():
#     conversation_history.clear()
#     return jsonify({"status": "cleared"})

# threading.Thread(target=lambda: flask_app.run(port=5000, use_reloader=False), daemon=True).start()
# print("✅ Flask running at http://127.0.0.1:5000")

✅ Flask running at http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


In [16]:
# # CELL B — Public URL via ngrok  (free, no firewall issues)
# # localtunnel had firewall errors — ngrok is much more reliable.

# # Step 1: install
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)

# # Step 2: sign up free at https://ngrok.com → copy your authtoken
# NGROK_TOKEN = "3AYCAM5xfC8hHB9OIjNjaI7tSxh_7H2Fx9UVdTWZ8dj3NgRNW"   # ← edit this line

# from pyngrok import ngrok, conf
# conf.get_default().auth_token = NGROK_TOKEN

# # Step 3: open tunnel
# tunnel = ngrok.connect(5000, bind_tls=True)
# public_url = tunnel.public_url
# print(f"\n🌐 Public URL: {public_url}")
# print(f"\n👉 Steps:")
# print(f"   1. Open  index.html  in your browser")
# print(f"   2. In the sidebar, paste this URL into the Backend URL box: {public_url}")
# print(f"   3. Click Set — you should see 'Connected ✓'")
# print(f"   4. Start chatting!\n")
# print(f"   (URL stays alive as long as this notebook cell is running)")


🌐 Public URL: https://yoko-overrepresentative-santos.ngrok-free.dev

👉 Steps:
   1. Open  index.html  in your browser
   2. In the sidebar, paste this URL into the Backend URL box: https://yoko-overrepresentative-santos.ngrok-free.dev
   3. Click Set — you should see 'Connected ✓'
   4. Start chatting!

   (URL stays alive as long as this notebook cell is running)


### ALTERNATIVE: If you don't want ngrok, use this instead of CELL B:
Open a NEW terminal / Anaconda Prompt and run: python -m http.server 8080 \
Then open:  http://localhost:8080/index.html \
And set Backend URL to:  http://127.0.0.1:5000 \
(works only for local demos, not for sharing a public URL)

In [48]:
# Run this ONCE to start the Flask server in background
import threading, json
from flask import Flask, request, jsonify
from flask_cors import CORS

flask_app = Flask(__name__)
CORS(flask_app)

@flask_app.route("/chat", methods=["POST"])
def chat():
    data  = request.get_json(force=True)
    query = data.get("message", "").strip()
    if not query:
        return jsonify({"error": "Empty message"}), 400
    try:
        answer, sources, queries = ask(query, verbose=False)   # ← your existing function
        src_list = [{"source": s["source"], "score": round(s["score"], 3)} for s in (sources or [])[:5]]
        turns    = len(conversation_history) // 2
        return jsonify({"answer": answer, "sources": src_list, "queries": queries or [], "turns": turns})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@flask_app.route("/clear", methods=["POST"])
def clear():
    conversation_history.clear()
    return jsonify({"ok": True})

@flask_app.route("/status", methods=["GET"])
def status():
    return jsonify({"turns": len(conversation_history) // 2, "max": MAX_TURNS})

PORT = 5050

def _run():
    flask_app.run(port=PORT, debug=False, use_reloader=False)

t = threading.Thread(target=_run, daemon=True)
t.start()
print(f"✅ StrategyAI server running at http://localhost:{PORT}")


# Display inside notebook as iframe  (optional)
# from IPython.display import IFrame, display
# display(IFrame("strategyai.html", width="100%", height=720))

# ── OR open in a real browser tab ──────────────────────────────────
import webbrowser, pathlib
webbrowser.open(pathlib.Path("strategyai.html").resolve().as_uri())

✅ StrategyAI server running at http://localhost:5050
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit


True